# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os ; import sys
sys.path.insert(0, os.path.abspath(os.path.join('./lib')))

import utilities
import detect_ignition
import harmonics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 128

import traceback

from scipy.signal import butter, filtfilt, hilbert, welch, csd, get_window

In [ ]:
ELECTRODES = ['EEG.AF3','EEG.AF4','EEG.F7','EEG.F8','EEG.F3','EEG.F4','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2']
RANGES  = {'Delta':[1,4],'Theta':[4,8],'Alpha':[8,12],'BetaL':[12,16], 'BetaH':[16,25],'Gamma':[25,45]}
FS = 128

# Load EEG Data

In [ ]:
FILENAME = "data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv"

RECORDS = utilities.load_eeg_csv(FILENAME, electrodes=ELECTRODES)

# Estimate Schumann Harmonics

In [ ]:
HARMONICS = harmonics.estimate_sr_harmonics(RECORDS, sr_channel='EEG.F4', fs=None,
                          f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
                          search_halfband=1.0, nperseg_sec=32.0, overlap=0.5)

SUBHARMONICS = harmonics.estimate_sr_harmonics(RECORDS, sr_channel='EEG.F4', fs=None,
                          f_can=(HARMONICS[0],HARMONICS[0]/2.0,HARMONICS[0]/3.0,HARMONICS[0]/4.0,HARMONICS[0]/5.0, HARMONICS[0]/6.0, HARMONICS[0]/7.0, HARMONICS[0]/8.0),
                          search_halfband=0.1, nperseg_sec=32.0, overlap=0.5)

# Detect Ignition

In [ ]:
# def detect_ignitions_session(
#     RECORDZ: pd.DataFrame,
#     sr_channel: Optional[str] = "EEG.F4",                 # kept for continuity in plots/logs
#     eeg_channels: Optional[List[str]] = None,
#     time_col: str = 'Timestamp',
#     out_dir: str = 'exports_ignitions/S01',
#     # detection params
#     center_hz: float = 7.83, half_bw_hz: float = 0.6,
#     smooth_sec: float = 0.25, z_thresh: float = 2.5,
#     min_isi_sec: float = 2.0, window_sec: float = 20.0, merge_gap_sec: float = 5.0,
#     # validation params (session-level R)
#     R_band: Tuple[float, float] = (8,13), R_win_sec: float = 1.0, R_step_sec: float = 0.25,
#     eta_pre_sec: float = 10.0, eta_post_sec: float = 10.0,
#     # NEW adaptive/event-centric knobs
#     sr_reference: str = 'auto-SSD',       # 'F4' | 'auto-SSD' | 'auto-PLV' | 'auto-PCA'
#     seed_method: str = 'latency',         # 'latency' | 'PGD'
#     pel_band: Tuple[float,float] = (60, 90),
#     electrode_xy: Optional[Dict[str, Tuple[float,float]]] = None,
#     harmonics: Tuple[int,...] = (2,3,4,5,6,7),
#     make_passport: bool = True,
#     show: bool = True,
#     verbose: bool = True
# ) -> Tuple[Dict[str, object], List[Tuple[int,int]]]:

import detect_ignition

out, IGNITION_WINDOWS = detect_ignition.detect_ignitions_session(
    RECORDS, eeg_channels=ELECTRODES,
    center_hz=HARMONICS[0], half_bw_hz=0.35,
    z_thresh=3,
    R_band=(HARMONICS[0]-0.5,HARMONICS[0]+0.5),
    sr_reference='auto-SSD', seed_method='latency',
    pel_band=(35,58), 
    harmonics_hz=HARMONICS,
    eta_pre_sec = 5.0, eta_post_sec = 5.0,
    out_dir='exports_ignitions/S01'
)

# Delta 

In [ ]:
# 1) Get crest peaks across events
events_df = out['result']['events'] if 'result' in out else out['events']
DF, _ = detect_ignition.summarize_delta_hotspots(
    RECORDS, events_df,
    eeg_channels=ELECTRODES, #['EEG.F3','EEG.F4','EEG.AF3','EEG.AF4','EEG.F7','EEG.F8','EEG.O1','EEG.O2'],
    combine='mean',
    crest_win=20,
    baseline_offset=(-10, -5),
    f_lo=1.5, f_hi=4,
    top_n=10
)

# 2) Cluster + print
hotspots = detect_ignition.cluster_delta_hotspots_meanshift(DF, z_thresh=2.0, bandwidth_quantile=0.2, fallback_bw=0.05)
pd.options.display.float_format = '{:0.3f}'.format
print("\nDelta surge hotspots (MeanShift, z≥2):")
display(hotspots)


# Animations

In [ ]:
IGN = 5

In [ ]:
anim, path = detect_ignition.animate_delta_psd(
    RECORDS,
    eeg_channels=ELECTRODES, #['EEG.F3','EEG.F4','EEG.AF3','EEG.AF4','EEG.F7','EEG.F8'],
    combine='mean',
    t_range=(IGNITION_WINDOWS[IGN][0],IGNITION_WINDOWS[IGN][1]),
    f_lo=0.5, f_hi=12,
    win_sec=10, step_sec=0.1,
    norm='z', baseline_range=(30,58),
    fps= 48,
    out_path='delta_psd_anim.mp4',
    show_inline=True, fill_alpha=0.15, dyn_ylim=False, ylim_pad=10,            # <-- 10% headroom        # <-- add soft shading
    title='Delta PSD (z-score) — frontal mean'
)

from IPython.display import HTML
HTML(anim.to_jshtml())   # inline without ffmpeg


In [ ]:
anim, path = detect_ignition.animate_rbp(
    RECORDS,
    eeg_channels=ELECTRODES, #['EEG.F3','EEG.F4','EEG.AF3','EEG.AF4','EEG.F7','EEG.F8'],
    combine='mean',     # or 'mean'/'median'
    t_range=(IGNITION_WINDOWS[IGN][0],IGNITION_WINDOWS[IGN][1]),
    win_sec=1.0, step_sec=0.04,
    fps=24,
    out_path='rbp_F4_lines_combined.mp4',
    show_inline=True,
    view_sec=5.0,         # fixed sliding window
    fill_alpha=0.15
)
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 64 

from IPython.display import HTML
HTML(anim.to_jshtml()) 

In [ ]:
sr_halfband = 0.5
sr_bands = [
    ('SR1', (HARMONICS[0]-sr_halfband,HARMONICS[0]+sr_halfband)), 
    ('2x', (HARMONICS[1]-sr_halfband,HARMONICS[1]+sr_halfband)), 
    ('3x', (HARMONICS[2]-sr_halfband,HARMONICS[2]+sr_halfband)), 
    ('4x', (HARMONICS[3]-sr_halfband,HARMONICS[3]+sr_halfband)),               
    ('5x', (HARMONICS[4]-sr_halfband,HARMONICS[4]+sr_halfband)), 
    ('6x', (HARMONICS[5]-sr_halfband,HARMONICS[5]+sr_halfband)), 
    ('7x', (HARMONICS[6]-sr_halfband,HARMONICS[6]+sr_halfband)), 
    ('8x', (HARMONICS[7]-sr_halfband,HARMONICS[7]+sr_halfband))
]

IGN = 5
anim, path, last_png = detect_ignition.animate_psd_stacked(
    RECORDS,
    eeg_channels=ELECTRODES, #['EEG.F4','EEG.FC6','EEG.P8'],
    combine='mean',
    t_range=(IGNITION_WINDOWS[IGN][0],IGNITION_WINDOWS[IGN][1]),
    default_bands='delta6',
    # default_bands='schumann',
    # bands=sr_bands,
    win_sec=3, step_sec=0.05,
    fps=24, out_path='psd_stacked_schumann.mp4',
    show_inline=True,
    title='Stacked absolute power — Schumann bands (SR±0.5 Hz) : ',
    legend_outside=True,
    save_last_frame=True
)
from IPython.display import HTML
HTML(anim.to_jshtml())


# Batch

## Epoc X

In [ ]:
files = [
    'data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv',
    'data/Test_06.11.20_14.28.18.md.pm.bp.csv',
    'data/20201229_29.12.20_11.27.57.md.pm.bp.csv',
    'data/med_EPOCX_111270_2021.06.12T09.50.52.04.00.md.bp.csv',
    'data/binaural_EPOCX_111270_2021.06.17T10.04.52.04.00.md.bp.csv',   
    'data/hyp_02.01.21_13.51.16.md.pm.bp.csv'
    # 'data/Quality Assessment_MM_EPOCX_111270_2021.02.16T10.51.08.05.00.md.mc.pm.fe.bp.csv'
]

# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1
for fpath in files:

    records = utilities.load_eeg_csv(fpath, electrodes=ELECTRODES)
    if fpath == 'data/hyp_02.01.21_13.51.16.md.pm.bp.csv':
        records = records.iloc[7680:-3840].reset_index(drop=True).copy()
    
    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)
    try:
        print(f"\n=== Processing {session_name} ===")

        out, ign_windows = detect_ignition.detect_ignitions_session(
            records, eeg_channels=ELECTRODES,
            z_thresh=3,R_band=(harms[0]-0.4,harms[0]+0.4),
            sr_reference='auto-SSD', seed_method='latency',
            pel_band=(35,58), 
            harmonics_hz=harms,
            eta_pre_sec = 5.0, eta_post_sec = 5.0,
            out_dir='exports_ignitions_batch/EPOCX/S'+ str(sc)
        )
        
        # DELTA PEAKS
        # 1) Get crest peaks across events
        events_df = out['result']['events'] if 'result' in out else out['events']
        DF, _ = detect_ignition.summarize_delta_hotspots(
            records, events_df,
            eeg_channels=ELECTRODES, #['EEG.F3','EEG.F4','EEG.AF3','EEG.AF4','EEG.F7','EEG.F8','EEG.O1','EEG.O2'],
            combine='median',
            crest_win=60,
            baseline_offset=(-10, -5),
            f_lo=1.5, f_hi=4,
            top_n=10
        )

        # 2) Cluster + print
        hotspots = detect_ignition.cluster_delta_hotspots_meanshift(DF, z_thresh=2.0, bandwidth_quantile=0.2, fallback_bw=0.05)
        pd.options.display.float_format = '{:0.3f}'.format
        print("\nDelta surge hotspots (MeanShift, z≥2):")
        display(hotspots)
        
        print("\n")
       
        
        ign=1
        for ign_win in ign_windows:
            anim, path, last_png = detect_ignition.animate_psd_stacked(
                records,
                eeg_channels=ELECTRODES, #['EEG.F4','EEG.FC6','EEG.P8'],
                combine='mean',
                t_range=(ign_win[0],ign_win[1]),
                default_bands='canonical',
                # default_bands='schumann',
                # bands=sr_bands,
                win_sec=3, step_sec=0.05,
                fps=24, out_path="exports_ignitions_batch/EPOCX/S"+ str(sc)+ "/" + "psd_stacked_canonical_epocx_s"+str(sc)+"e"+str(ign)+".mp4",
                show_inline=True,
                title='Stacked absolute power — canonical bands : ',
                legend_outside=True,
                save_last_frame=True,
                last_frame_path="exports_ignitions_batch/EPOCX/S"+ str(sc)+ "/" + "psd_stacked_canonical_epocx_s"+str(sc)+"e"+str(ign)+".png"
            )
            ign = ign + 1
        sc = sc + 1
                
        # Store summary row
        summ = out['summary'].copy()
        summ['session'] = session_name[:30]+" ..."
        summ['n_events'] = summ.get('n_events', 0)
        master_rows.append(summ)

    except Exception as e:
        print(f"[ERROR] {session_name}: {e}")
        traceback.print_exc()
        # add a failed row so you keep the log complete
        master_rows.append({'session': session_name[:20]+" ...", 'n_events': np.nan, 'error': str(e)})

# 5) Save master summary across sessions
master_df = pd.DataFrame(master_rows)
master_csv = os.path.join(ROOT_OUT, 'master_ignition_summary-EPOCX.csv')
master_df.to_csv(master_csv, index=False)

print("\n=== Batch complete ===")
print("Master summary saved to:", master_csv)
print(master_df.fillna('').to_string(index=False))

## Insight

In [ ]:
INSIGHT_ELECTRODES = ['EEG.AF3','EEG.AF4','EEG.T7','EEG.T8', 'EEG.Pz']

# 1) List your input files (CSV paths)
insight_files = [
    'data/testing_INSIGHT2_111270_2024.02.10T10.25.53.06.00.md.pm.bp.csv',
    'data/shit_INSIGHT2_111270_2024.02.10T13.00.17.06.00.md.pm.bp.csv',
    'data/testing_INSIGHT2_111270_2024.02.12T10.25.21.06.00.md.pm.bp.csv',
    'data/testing_INSIGHT2_111270_2024.02.13T08.26.34.06.00.md.pm.bp.csv',
    'data/testing_INSIGHT2_111270_2024.02.15T11.43.15.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.01T11.19.02.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.02T10.03.06.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.02T10.17.57.06.00.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.05T07.14.31.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.05T07.02.55.06.00.bp.csv',
    'data/tet_INSIGHT2_111270_2024.03.05T07.49.33.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.08T12.26.06.06.00.md.pm.bp.csv'
]


In [ ]:
# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1
for fpath in insight_files:
    records = utilities.load_eeg_csv(fpath, electrodes=INSIGHT_ELECTRODES)
    records = records.iloc[3840:-1920].reset_index(drop=True).copy()
    
    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)
    try:
        print(f"\n=== Processing {session_name} ===\n")

        harms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.AF4', fs=None,
                    f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
                    search_halfband=0.8, nperseg_sec=32.0, overlap=0.5)

        subharms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.AF4', fs=None,
                          f_can=(harms[0]/2.0,harms[0]/3.0,harms[0]/4.0,harms[0]/5.0, harms[0]/6.0, harms[0]/7.0, harms[0]/8.0),
                          search_halfband=0.1, nperseg_sec=32.0, overlap=0.5)

        
        
        formatted_list_fstring = [f"{num:.2f}" for num in harms]
        formatted_list_subfstring = [f"{num:.2f}" for num in subharms]
        print(f"Estimate SR harmonics: {formatted_list_fstring}")
       
        

        out, ign_windows = detect_ignition.detect_ignitions_session(
            records, eeg_channels=INSIGHT_ELECTRODES,
            z_thresh=3,R_band=(harms[0]-0.4,harms[0]+0.4),
            sr_channel="EEG.Pz", sr_reference='auto-SSD', seed_method='latency',
            pel_band=(35,58),
            harmonics_hz=harms,
            eta_pre_sec = 5.0, eta_post_sec = 5.0,
            out_dir='exports_ignitions_batch/INSIGHT/S'+ str(sc)
        )

        # 1) Get crest peaks across events
        events_df = out['result']['events'] if 'result' in out else out['events']
        DF, _ = detect_ignition.summarize_delta_hotspots(
            records, events_df,
            eeg_channels=['EEG.AF3','EEG.AF4', 'EEG.Pz'],
            combine='median',
            crest_win=60,
            baseline_offset=(-10, -5),
            f_lo=1.5, f_hi=4,
            top_n=10
        )

        print(f"\nEstimate SR subharmonics: {formatted_list_subfstring}")
        
        # 2) Cluster + print
        hotspots = detect_ignition.cluster_delta_hotspots_meanshift(DF, z_thresh=2.0, bandwidth_quantile=0.2, fallback_bw=0.05)
        pd.options.display.float_format = '{:0.3f}'.format
        print("\nDelta surge hotspots (MeanShift, z≥2):")
        display(hotspots)



        ign=1
        for ign_win in ign_windows:
            anim, path, last_png = detect_ignition.animate_psd_stacked(
                records,
                eeg_channels=INSIGHT_ELECTRODES, #['EEG.F4','EEG.FC6','EEG.P8'],
                combine='mean',
                t_range=(ign_win[0],ign_win[1]),
                default_bands='canonical',
                # default_bands='schumann',
                # bands=sr_bands,
                win_sec=3, step_sec=0.05,
                fps=24, out_path="exports_ignitions_batch/INSIGHT/S"+ str(sc)+ "/" + "psd_stacked_canonical_insight_s"+str(sc)+"e"+str(ign)+".mp4",
                show_inline=True,
                title='Stacked absolute power — canonical bands : ',
                legend_outside=True,
                save_last_frame=True,
                last_frame_path="exports_ignitions_batch/INSIGHT/S"+ str(sc)+ "/" + "psd_stacked_canonical_insight_s"+str(sc)+"e"+str(ign)+".png"
            )
            ign = ign + 1
        
        sc = sc+1

        
        # Store summary row
        summ = out['summary'].copy()
        summ['session'] = session_name[:30]+" ..."
        summ['n_events'] = summ.get('n_events', 0)
        master_rows.append(summ)



    except Exception as e:
        print(f"[ERROR] {session_name}: {e}")
        traceback.print_exc()
        # add a failed row so you keep the log complete
        master_rows.append({'session': session_name[:20]+" ...", 'n_events': np.nan, 'error': str(e)})

# 5) Save master summary across sessions
master_df = pd.DataFrame(master_rows)
master_csv = os.path.join(ROOT_OUT, 'master_ignition_summary-INSIGHT.csv')
master_df.to_csv(master_csv, index=False)

print("\n=== Batch complete ===")
print("Master summary saved to:", master_csv)
print(master_df.fillna('').to_string(index=False))

## Muse

In [ ]:
import utilities

MUSE_ELECTRODES = ['EEG.AF7','EEG.AF8','EEG.TP9','EEG.TP10']

# 1) List your input files (CSV paths)
muse_files = [
    # 'data/test.csv',
    'data/Muse-461E_2019-11-28--16-01-03_1575083390915.csv',
    'data/Muse-461E_2019-11-27--19-26-58_1575321084010.csv',
    'data/Muse-461E_2019-12-03--12-47-46_1575407413023.csv',
    # 'data/Muse-461E_2019-11-20--21-13-30_1574381977511.csv',
    'data/Muse-461E_2019-12-03--20-53-25_1575436553405.csv',
    'data/Muse-461E_2019-12-04--12-05-39_1575491284751.csv',
    'data/Muse-461E_2019-12-05--12-17-34_1575579123713.csv',
    'data/Muse-461E_2019-12-06--14-36-37_1575672417939.csv',
    'data/Muse-461E_2019-12-06--20-54-24_1575694700892.csv',
    'data/Muse-461E_2019-12-06--20-57-59_1575694836983.csv',
    # 'data/Muse-461E_2019-12-06--21-00-26_1575696038038.csv',
    'data/Muse-461E_2019-12-07--13-11-51_1575754159523.csv',
    'data/Muse-461E_2019-12-07--18-28-16_1575772914160.csv',
    'data/Muse-461E_2019-12-15--15-20-11_1576454995317.csv',
    'data/Muse-357D_2019-12-17--14-20-44_1576623304487.csv'
#     'data/Muse-357D_2019-12-24--10-52-16_1577215680302.csv',
#     'data/Muse-461E_2020-07-12--13-30-04_1594576299757.csv'
]

# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1
for fpath in muse_files:
    records = utilities.load_eeg_csv(fpath, electrodes=MUSE_ELECTRODES,device='muse')
    records = records.iloc[7680:-3840].reset_index(drop=True).copy()

    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)
    try:
        print(f"\n=== Processing {session_name} ===\n")

        harms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.AF8', fs=None,
                    f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
                    search_halfband=0.8, nperseg_sec=32.0, overlap=0.5)

        formatted_list_fstring = [f"{num:.2f}" for num in harms]
        print(f"Estimate SR harmonics: {formatted_list_fstring}")
        

        out, ign_windows = detect_ignition.detect_ignitions_session(
            records, eeg_channels=MUSE_ELECTRODES,
            z_thresh=3,R_band=(harms[0]-0.4,harms[0]+0.4),
            sr_channel="EEG.AF8", sr_reference='auto-SSD', seed_method='latency',
            pel_band=(35,58),
            harmonics_hz=harms,
            eta_pre_sec = 5.0, eta_post_sec = 5.0,
            out_dir='exports_ignitions_batch/MUSE/S'+ str(sc)
        )

        ign=1
        for ign_win in ign_windows:
            anim, path, last_png = detect_ignition.animate_psd_stacked(
                records,
                eeg_channels=MUSE_ELECTRODES, #['EEG.F4','EEG.FC6','EEG.P8'],
                combine='mean',
                t_range=(ign_win[0],ign_win[1]),
                default_bands='canonical',
                # default_bands='schumann',
                # bands=sr_bands,
                win_sec=3, step_sec=0.05,
                fps=24, out_path="exports_ignitions_batch/MUSE/S"+ str(sc)+ "/" + "psd_stacked_canonical_muse_s"+str(sc)+"e"+str(ign)+".mp4",
                show_inline=True,
                title='Stacked absolute power — canonical bands : ',
                legend_outside=True,
                save_last_frame=True,
                last_frame_path="exports_ignitions_batch/MUSE/S"+ str(sc)+ "/" + "psd_stacked_canonical_muse_s"+str(sc)+"e"+str(ign)+".png"
            )
            ign = ign + 1
        
        print("\n")
        sc = sc+1

        
        # Store summary row
        summ = out['summary'].copy()
        summ['session'] = session_name[:30]+" ..."
        summ['n_events'] = summ.get('n_events', 0)
        master_rows.append(summ)



    except Exception as e:
        print(f"[ERROR] {session_name}: {e}")
        traceback.print_exc()
        # add a failed row so you keep the log complete
        master_rows.append({'session': session_name[:20]+" ...", 'n_events': np.nan, 'error': str(e)})

# 5) Save master summary across sessions
master_df = pd.DataFrame(master_rows)
master_csv = os.path.join(ROOT_OUT, 'master_ignition_summary-MUSE.csv')
master_df.to_csv(master_csv, index=False)

print("\n=== Batch complete ===")
print("Master summary saved to:", master_csv)
print(master_df.fillna('').to_string(index=False))